In [0]:
base_adls2_path = "abfss://customer360unitycatalog@kmstorage9490.dfs.core.windows.net"

In [0]:
dbutils.widgets.dropdown("schema", "select", ['select',"bronze", "silver",'gold'], "Select schema to drop?")
dbutils.widgets.dropdown("container_name", "select", ['select',"bronze", "silver",'gold'], "Select conatiner to drop?")

In [0]:
# ==========================================
# DROP ALL TABLES IN SCHEMA
# ==========================================

catalog_name = "customer360_cata"
schema_name = dbutils.widgets.get('schema')
if schema_name == "select":
    print("Please select a schema to drop tables from")
else:    
    tables = spark.sql(
        f"SHOW TABLES IN {catalog_name}.{schema_name}"
    ).collect()

    for table in tables:

        table_name = table.tableName

        full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

        print(f"Dropping: {full_table_name}")

        spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")

    print("All tables dropped successfully")

In [0]:
container_name = dbutils.widgets.get('container_name')
if container_name == "select":
    print("Please select a container to drop")
else:
    print(f"Dropping: {base_adls2_path}/{container_name}")
    adls2_container_folders = dbutils.fs.ls(f"{base_adls2_path}/{container_name}")
    for folder in adls2_container_folders:
        dbutils.fs.rm(folder.path, recurse=True)

    adls2_checkpoints = dbutils.fs.ls(f"{base_adls2_path}/checkpoints")
    for folder in adls2_checkpoints:
        dbutils.fs.rm(folder.path, recurse=True)    

    adls2_schema = dbutils.fs.ls(f"{base_adls2_path}/schema")
    for folder in adls2_schema:
        dbutils.fs.rm(folder.path, recurse=True)        

In [0]:
%sql
--drop table if exists customer360_cata.silver.customers